In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ACCESSING THE DATA

In [ ]:
df = pd.read_csv("insurance.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
df.describe()

In [ ]:
df["sex"].value_counts()

In [ ]:
df["children"].value_counts()

In [ ]:
df["smoker"].value_counts()

In [ ]:
df["region"].value_counts()

EXPLORATORY DATA ANALYSIS(EDA)

In [ ]:
sns.histplot(data=df["charges"],kde=True)

In [ ]:
sns.histplot(df["age"], kde=True)

In [ ]:
sns.histplot(df["bmi"], kde=True)

In [ ]:
sns.countplot(x="children", data=df)

In [ ]:
sns.boxplot(x=df["age"])

In [ ]:
sns.boxplot(x=df["bmi"])

In [ ]:
sns.boxplot(x=df["charges"])

In [ ]:
df.corr(numeric_only=True)

In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot=True,cmap="YlGnBu")

In [ ]:
sns.scatterplot(data=df,x="age",y="charges")

In [ ]:
sns.scatterplot(data=df,x="bmi",y="charges")

In [ ]:
sns.boxplot(x="sex",y="charges",data=df)

In [ ]:
sns.boxplot(x="smoker",y="charges",data=df)

In [ ]:
sns.boxplot(x="region",y="charges",data=df)

In [ ]:
sns.pairplot(df,vars=["age","bmi","children","charges"])

In [ ]:
sns.scatterplot(data=df,x="age",y="charges",hue="smoker")

In [ ]:
df["charges"].skew()

In [ ]:
df["smoker"]=(df["smoker"]=="yes").astype(int)

In [ ]:
df=pd.get_dummies(df,columns=["sex","region"],dtype=int)

In [ ]:
df

In [ ]:
X=df.drop("charges",axis=1)
y=df["charges"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split( X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler


normal_model = Pipeline([
("scaler",StandardScaler()),
("model",LinearRegression())
])

In [ ]:
normal_model.fit(X_train,y_train)
y_pred=normal_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
mean_absolute_error(y_test,y_pred)

In [ ]:
mean_squared_error(y_test,y_pred)

In [ ]:
r2_score(y_test,y_pred)

In [ ]:
r2_score(y_train,normal_model.predict(X_train))

In [ ]:
r2_score(y_test,normal_model.predict(X_test))

In [ ]:
normal_model.named_steps["model"].coef_

In [ ]:
print(normal_model.score(X_train, y_train))
print(normal_model.score(X_test, y_test))

In [ ]:
residuals=y_test-y_pred

In [ ]:
sns.histplot(residuals,kde=True)
plt.xlabel("Residuals")

In [ ]:
sns.histplot(np.abs(residuals),kde=True)
plt.xlabel("Residuals")

In [ ]:
residuals.mean()

In [ ]:
plt.scatter(y_pred, residuals)

In [ ]:
absolute_error=np.abs(residuals)

In [ ]:
error_df=pd.DataFrame({"Actual":y_test,"Predicted":y_pred,"Residual":residuals,"absolute error":absolute_error})
error_df.sort_values(by="absolute error",ascending=False).head(15)

In [ ]:
normal_model.named_steps["model"]

In [ ]:
from sklearn.linear_model import Ridge

pipe = Pipeline([
("scaler",StandardScaler()),
("model",Ridge())
])
param_grid = {
    "model__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}
scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error"
}

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(pipe,param_grid,cv=5,scoring=scoring,refit="r2")
grid.fit(X_train,y_train)

In [ ]:
ridge_best_model=grid.best_estimator_

In [ ]:
y_pred = ridge_best_model.predict(X_test)

In [ ]:
print("r2 =",r2_score(y_test,y_pred))
print("mae =",mean_absolute_error(y_test,y_pred))
print("mse =",mean_squared_error(y_test,y_pred))

In [ ]:
print(grid.score(X_train, y_train))
print(grid.score(X_test, y_test))

In [ ]:
ridge_best_model.named_steps["model"].coef_

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient":grid.best_estimator_.named_steps["model"].coef_
})

print(coef_df)

In [ ]:
results=pd.DataFrame(grid.cv_results_)
results

In [ ]:
results["mean_test_mae"] = -results["mean_test_mae"]
results["mean_test_mse"] = -results["mean_test_mse"]

In [ ]:
results[["param_model__alpha","mean_test_r2","mean_test_mae","mean_test_mse"]]

In [ ]:
from sklearn.linear_model import Lasso


pipe = Pipeline([
    ("Scaler", StandardScaler()),
    ("model", Lasso(max_iter=10000))
])
param_grid = {
    "model__alpha":[0.001,0.01,0.1,1,10,100]}
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="r2"
)

grid.fit(X_train,y_train)

In [ ]:
lasso_best_model=grid.best_estimator_

In [ ]:
y_pred = lasso_best_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error,r2_score

print("MAE =",mean_absolute_error(y_test,y_pred))
print("R² =",r2_score(y_test,y_pred))

In [ ]:
grid.best_estimator_.named_steps["model"].coef_

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": grid.best_estimator_.named_steps["model"].coef_
})

coef_df["coef_rounded"] = coef_df["Coefficient"].round(2)

print(coef_df)

In [ ]:
coef_df[coef_df["coef_rounded"]!=0]

In [ ]:
param_grid = [
    {
        "model": [LinearRegression()]
    },
    {
        "model": [Ridge()],
        "model__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
    },
    {
        "model": [Lasso(max_iter=10000)],
        "model__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
    }
]
scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error"
}
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring=scoring,
    refit="r2",
    n_jobs=-1
)

grid.fit(X_train, y_train)

In [ ]:
combined_best_model_linear_ridge_lasso_without_poly_features=grid.best_estimator_

In [ ]:
results = pd.DataFrame(grid.cv_results_)

results["mean_test_mae"] = -results["mean_test_mae"]
results["mean_test_mse"] = -results["mean_test_mse"]

results_table = results[
    [
        "param_model",
        "param_model__alpha",
        "mean_test_r2",
        "mean_test_mae",
        "mean_test_mse"
    ]
]

results_table.sort_values("mean_test_r2", ascending=False)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", Ridge())
])
param_grid = [
    {
        "model": [LinearRegression()]
    },
    {
        "poly__degree": [1, 2, 3],
        "model": [Ridge()],
        "model__alpha": [0.01, 0.1, 1, 10, 100]
    },
    {
        "poly__degree": [1, 2, 3],
        "model": [Lasso(max_iter=10000)],
        "model__alpha": [0.01, 0.1, 1, 10, 100]
    }
]
scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error"
}
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring=scoring,
    refit="r2",
    n_jobs=-1
)

grid.fit(X_train, y_train)

In [ ]:
import pandas as pd

results = pd.DataFrame(grid.cv_results_)

results["mean_test_mae"] = -results["mean_test_mae"]
results["mean_test_mse"] = -results["mean_test_mse"]

results_table = results[
    [
        "param_poly__degree",
        "param_model",
        "param_model__alpha",
        "mean_test_r2",
        "mean_test_mae",
        "mean_test_mse"
    ]
]

results_table.sort_values("mean_test_r2", ascending=False)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("R²:", r2_score(y_test, y_pred))

In [ ]:
print("Train R²:", best_model.score(X_train, y_train))
print("Test R²:", best_model.score(X_test, y_test))

In [ ]:
feature_names = best_model.named_steps["poly"].get_feature_names_out(X.columns)
coef = best_model.named_steps["model"].coef_
coef_df = pd.DataFrame({"Feature": feature_names, "Coefficient": coef})
coef_df["coef_rounded"] = coef_df["Coefficient"].round(2)
coef_df.sort_values("Coefficient",key=abs,ascending=False).head(20)

In [ ]:
coef_df[coef_df["coef_rounded"]!=0]

In [ ]:
from sklearn.model_selection import LearningCurveDisplay

LearningCurveDisplay.from_estimator(
    best_model,
    X_train,
    y_train,
    cv=5,
    scoring="r2",
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

plt.show()

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ValidationCurveDisplay
import matplotlib.pyplot as plt
import numpy as np

pipe = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10))
])
ValidationCurveDisplay.from_estimator(
    pipe,
    X_train,
    y_train,
    param_name="poly__degree",
    param_range=[1, 2, 3, 4],
    cv=5,
    scoring="r2",
    n_jobs=-1
)

plt.show()

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])
ValidationCurveDisplay.from_estimator(
    pipe,
    X_train,
    y_train,
    param_name="model__alpha",
    param_range=np.logspace(-3, 3, 7),
    cv=5,
    scoring="r2",
    n_jobs=-1
)

plt.xscale("log")
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(random_state=42)

param_grid = {
    "max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10]
}

scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error"
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring=scoring,
    refit="r2",
    n_jobs=-1
)

grid.fit(X_train, y_train)

In [ ]:
decision_tree_best_model = grid.best_estimator_

In [ ]:
y_pred = grid.predict(X_test)

print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("Test R²:", r2_score(y_test, y_pred))

In [ ]:
print("Train R²:", grid.score(X_train, y_train))
print("Test R²:", grid.score(X_test, y_test))

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance":decision_tree_best_model.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 10))

plot_tree(
    decision_tree_best_model,
    feature_names=X.columns,
    filled=True,
    rounded=True
)

plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=1
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", None]
}

scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error"
}

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring=scoring,
    refit="r2",
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

In [ ]:
random_forest_best_model = grid.best_estimator_

In [ ]:
y_pred = grid.predict(X_test)

print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("Test R²:", r2_score(y_test, y_pred))

In [ ]:
print("Train R²:", grid.score(X_train, y_train))
print("Test R²:", grid.score(X_test, y_test))

In [ ]:
results = pd.DataFrame(grid.cv_results_)

results["mean_test_mae"] = -results["mean_test_mae"]
results["mean_test_rmse"] = -results["mean_test_rmse"]

results_table = results[
    [
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_leaf",
        "param_max_features",
        "mean_test_r2",
        "mean_test_mae",
        "mean_test_rmse"
    ]
].sort_values("mean_test_r2", ascending=False)

results_table.head(10)

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest_best_model.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
PartialDependenceDisplay.from_estimator(
    grid,
    X_train,
    features=["age", "bmi", "smoker"],
    kind="average"
)

plt.show()

In [ ]:
PartialDependenceDisplay.from_estimator(
    random_forest_best_model,
    X_train,
    features=[("bmi", "smoker")],
    kind="average"
)

plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

model_list = [
    ("Normal Linear Regression", normal_model),
    ("Ridge (tuned)", ridge_best_model),
    ("Lasso (tuned)", lasso_best_model),
    ("Linear/Ridge/Lasso (no poly)", combined_best_model_linear_ridge_lasso_without_poly_features),
    ("Polynomial Linear/Ridge/Lasso", best_model),
    ("Decision Tree", decision_tree_best_model),
    ("Random Forest", random_forest_best_model)
]

final_results = []

for name, model in model_list:
    y_pred = model.predict(X_test)

    final_results.append({
        "Model": name,
        "Train R²": model.score(X_train, y_train),
        "Test R²": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred) ** 0.5
    })

final_results_df = pd.DataFrame(final_results)

final_results_df = final_results_df.sort_values("Test R²", ascending=False)

final_results_df

In [ ]:
final_results